# EDA: live win probability states

Tables only (no plots). The held-out season (2024-25) is dropped before every table that reads the outcome or fits. Every cell below runs on the synthetic fixtures of `tests/_fx_winprob.py`, so **every number is fixture output, not a result**. Point `load_data` at DATA_ROOT on the owner's Mac for the real tables.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
from analytics.winprob import features as F, eda
FIXTURES = True   # set False and DATA_ROOT below for the real run
DATA_ROOT = None
d = F.load_fixtures(0, n_games=20, seasons=('2020-21', '2021-22')) if FIXTURES else F.load_data(DATA_ROOT)
states = F.build_states(d['playbyplay'], d['games'], d['odds'], d.get('team_names'))
HELDOUT_SEASON = '2024-25'   # never read by a table that uses home_win or fits
train = eda.drop_heldout_season(states, HELDOUT_SEASON)
print('fixture output, not a result' if FIXTURES else 'real data')
print(states.shape, sorted(states['season'].unique()))

## Margin distribution by time bucket

Quantiles of the home-minus-away margin per bucket of minutes remaining in regulation.

In [ ]:
eda.margin_distribution(states)

## Time distribution

Rows per period and per time bucket (the late buckets hold fewer events because they span fewer minutes).

In [ ]:
eda.time_distribution(states)

## Win rate by margin bin and time bucket

The home win rate should move from near 0.5 at a tied score early to near 0 or 1 at large late margins.

In [ ]:
eda.win_rate_by_margin_bucket(train)

## Spread coverage

Share of games with a closing spread from the odds table and share of events with a known possession indicator.

In [ ]:
eda.spread_coverage(train)

## Shuffled-label check

A logistic regression on margin and time under grouped folds by game, once on the real labels and once on labels shuffled across games. The shuffled run should sit at the base-rate Brier; anything far below it would mean a feature carries the outcome.

In [ ]:
eda.shuffled_label_check(train, seed=0)